# CoinDCX Trade Summary - Optimized Single LLM Call

This notebook keeps the same core flow as `assessment_3.ipynb`, but uses one LLM call to detect the header row, detect the required column names, and produce qualitative sample insights. All numeric trade stats are computed deterministically in Python.

In [2]:
import json
import os
import re
from typing import Any, Dict, List, Optional

import pandas as pd
from dotenv import load_dotenv
from groq import Groq

SHEET_NAME = "Instant Orders"
MODEL_NAME = "openai/gpt-oss-120b"
REQUIRED_COLUMN_MAPPINGS = ["crypto", "side", "gross_amount", "avg_price"]

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY environment variable is not set. Please set it in your .env file.")

client = Groq(api_key=api_key)

print("Imports loaded successfully")
print("Groq client initialized")

Imports loaded successfully
Groq client initialized


In [3]:
def extract_json_from_text(text: str) -> Dict[str, Any]:
    if not text or not text.strip():
        raise ValueError("Model returned an empty response")

    cleaned = text.strip()
    fenced = re.search(r"```(?:json)?\s*(.*?)```", cleaned, re.DOTALL | re.IGNORECASE)
    if fenced:
        cleaned = fenced.group(1).strip()

    match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if not match:
        raise ValueError("Model did not return a JSON object")

    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError as exc:
        raise ValueError(f"Model returned invalid JSON: {exc}") from exc

    if not isinstance(parsed, dict):
        raise ValueError("Model JSON response must be an object")

    return parsed


def validate_single_llm_response(data: Dict[str, Any]) -> Dict[str, Any]:
    header_detection = data.get("header_detection")
    column_mapping = data.get("column_mapping")
    summary = data.get("summary", {})

    if not isinstance(header_detection, dict):
        raise ValueError("LLM response is missing header_detection object")

    header_row_index = header_detection.get("header_row_index")
    confidence = header_detection.get("confidence")
    reasoning = header_detection.get("reasoning", "")

    if not isinstance(header_row_index, int) or header_row_index < 0:
        raise ValueError("header_row_index must be a non-negative integer")

    if not isinstance(confidence, (int, float)) or not 0 <= float(confidence) <= 1:
        raise ValueError("confidence must be a number between 0 and 1")

    if not isinstance(column_mapping, dict):
        raise ValueError("LLM response is missing column_mapping object")

    normalized_mapping = {}
    for key in REQUIRED_COLUMN_MAPPINGS:
        value = column_mapping.get(key)
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f"column_mapping.{key} must be a non-empty string")
        normalized_mapping[key] = value.strip()

    if summary is None:
        summary = {}
    if not isinstance(summary, dict):
        raise ValueError("summary must be an object")

    key_insights = summary.get("key_insights", [])
    sample_notes = summary.get("sample_notes", [])

    if not isinstance(key_insights, list):
        key_insights = [str(key_insights)]
    if not isinstance(sample_notes, list):
        sample_notes = [str(sample_notes)]

    return {
        "header_detection": {
            "header_row_index": header_row_index,
            "confidence": float(confidence),
            "reasoning": str(reasoning),
        },
        "column_mapping": normalized_mapping,
        "summary": {
            "key_insights": [str(item) for item in key_insights if str(item).strip()],
            "sample_notes": [str(item) for item in sample_notes if str(item).strip()],
        },
    }

In [5]:
def read_raw_sheet(file_path: str, sheet_name: str = SHEET_NAME) -> pd.DataFrame:
    return pd.read_excel(file_path, sheet_name=sheet_name, header=None, engine="openpyxl")


def build_single_llm_prompt(raw_df: pd.DataFrame, preview_rows: int = 15, sample_data_rows: int = 20) -> str:
    rows_to_send = preview_rows + sample_data_rows

    with pd.option_context("display.max_rows", rows_to_send, "display.max_columns", None, "display.width", 20000, "display.max_colwidth", 200):
        raw_sample = raw_df.head(rows_to_send).to_string(index=True)

    return f"""
You are a data analyst reviewing a CoinDCX Excel sheet preview.

You must do three things in one response:
1. Identify the actual header row index using the 0-based row index shown on the left.
2. Detect the exact header names needed for deterministic trade calculations.
3. Provide short qualitative insights from only the visible sample rows.

Rules:
- The header row contains column names and is usually followed by trade data rows.
- The header row is not a title, metadata, timestamp, or blank row.
- column_mapping values must exactly match the visible column names in the detected header row after trimming whitespace.
- Map crypto to the coin, token, asset, or crypto column.
- Map side to the buy/sell direction column.
- Map gross_amount to the INR paid/received amount column.
- Map avg_price to the average buy/sell price column.
- Do not compute official totals, investment, average price, or buy/sell counts.
- Numeric stats will be calculated deterministically in Python after this response.
- Return JSON only. Do not include markdown or explanatory text outside JSON.

Required JSON schema:
{{
  "header_detection": {{
    "header_row_index": 0,
    "confidence": 0.0,
    "reasoning": "short explanation"
  }},
  "column_mapping": {{
    "crypto": "exact crypto column name",
    "side": "exact buy/sell side column name",
    "gross_amount": "exact gross INR amount column name",
    "avg_price": "exact average price column name"
  }},
  "summary": {{
    "key_insights": [
      "short qualitative insight based only on the provided sample"
    ],
    "sample_notes": [
      "short note about visible sample patterns or anomalies"
    ]
  }}
}}

Excel preview with raw row indexes:
{raw_sample}
""".strip()

In [6]:
def analyze_sheet_with_llm(raw_df: pd.DataFrame, preview_rows: int = 15, sample_data_rows: int = 20) -> Dict[str, Any]:
    prompt = build_single_llm_prompt(raw_df, preview_rows=preview_rows, sample_data_rows=sample_data_rows)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    raw_content = response.choices[0].message.content.strip()
    parsed = extract_json_from_text(raw_content)
    return validate_single_llm_response(parsed)

In [7]:
def clean_header_value(value: Any) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def make_unique_columns(columns: List[str]) -> List[str]:
    seen: Dict[str, int] = {}
    unique_columns = []

    for column in columns:
        count = seen.get(column, 0)
        unique_columns.append(column if count == 0 else f"{column}_{count}")
        seen[column] = count + 1

    return unique_columns


def normalize_column_label(value: str) -> str:
    return re.sub(r"\s+", " ", str(value).strip()).casefold()


def resolve_column_name(df: pd.DataFrame, detected_column: str, semantic_name: str) -> str:
    if detected_column in df.columns:
        return detected_column

    normalized_target = normalize_column_label(detected_column)
    matches = [column for column in df.columns if normalize_column_label(column) == normalized_target]

    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError(f"Detected {semantic_name} column is ambiguous: {detected_column}")

    raise ValueError(f"Detected {semantic_name} column was not found in the dataframe: {detected_column}")


def resolve_column_mapping(df: pd.DataFrame, column_mapping: Dict[str, str]) -> Dict[str, str]:
    return {
        key: resolve_column_name(df, column_mapping[key], key)
        for key in REQUIRED_COLUMN_MAPPINGS
    }


def load_clean_dataframe(raw_df: pd.DataFrame, header_row_index: int, crypto_column: str) -> pd.DataFrame:
    if header_row_index >= len(raw_df):
        raise ValueError("header_row_index is outside the dataframe range")

    header_values = [clean_header_value(value) for value in raw_df.iloc[header_row_index].tolist()]
    valid_positions = [index for index, value in enumerate(header_values) if value]

    if not valid_positions:
        raise ValueError("Detected header row does not contain valid column names")

    df = raw_df.iloc[header_row_index + 1:, valid_positions].copy()
    df.columns = make_unique_columns([header_values[index] for index in valid_positions])
    df = df.dropna(how="all")

    crypto_column = resolve_column_name(df, crypto_column, "crypto")
    crypto_values = df[crypto_column].astype("string").str.strip()
    df = df[df[crypto_column].notna() & crypto_values.ne("")]

    return df.reset_index(drop=True)

In [8]:
def require_columns(df: pd.DataFrame, required_columns: List[str]) -> None:
    missing_columns = [column for column in required_columns if column not in df.columns]
    if missing_columns:
        missing = ", ".join(missing_columns)
        raise ValueError(f"Clean dataframe is missing required columns: {missing}")


def compute_trade_stats(df: pd.DataFrame, column_mapping: Dict[str, str]) -> Dict[str, Any]:
    side_column = resolve_column_name(df, column_mapping["side"], "side")
    amount_column = resolve_column_name(df, column_mapping["gross_amount"], "gross_amount")
    price_column = resolve_column_name(df, column_mapping["avg_price"], "avg_price")

    require_columns(df, [side_column, amount_column, price_column])

    side_values = df[side_column].astype("string").str.strip().str.lower()
    buy_mask = side_values.eq("buy")
    sell_mask = side_values.eq("sell")

    amount_values = pd.to_numeric(df[amount_column], errors="coerce")
    price_values = pd.to_numeric(df[price_column], errors="coerce")

    total_buy_amount = amount_values[buy_mask].sum(min_count=1)
    avg_buy_price = price_values[buy_mask].mean()

    return {
        "total_trades": int(len(df)),
        "buy_trades": int(buy_mask.sum()),
        "sell_trades": int(sell_mask.sum()),
        "total_buy_amount": 0.0 if pd.isna(total_buy_amount) else float(total_buy_amount),
        "avg_buy_price": None if pd.isna(avg_buy_price) else float(avg_buy_price),
    }

In [9]:
def format_inr(value: Optional[float]) -> str:
    if value is None:
        return "N/A"
    return f"INR {value:,.2f}"


def build_trade_summary(stats: Dict[str, Any], llm_analysis: Dict[str, Any]) -> str:
    insights = llm_analysis.get("summary", {}).get("key_insights", [])
    notes = llm_analysis.get("summary", {}).get("sample_notes", [])
    insight_items = insights or notes or ["No additional sample insights were returned by the model."]

    insight_text = "\n".join(f"- {item}" for item in insight_items[:5])

    return f"""Trade Summary
- Total trades: {stats['total_trades']}
- Buy trades: {stats['buy_trades']}
- Sell trades: {stats['sell_trades']}
- Total investment on buy side: {format_inr(stats['total_buy_amount'])}
- Average buy price: {format_inr(stats['avg_buy_price'])}

Key insights from the sampled rows:
{insight_text}"""


def process_coindcx_file(
    file_path: str,
    sheet_name: str = SHEET_NAME,
    preview_rows: int = 15,
    sample_data_rows: int = 20,
) -> Dict[str, Any]:
    raw_df = read_raw_sheet(file_path, sheet_name=sheet_name)
    llm_analysis = analyze_sheet_with_llm(raw_df, preview_rows=preview_rows, sample_data_rows=sample_data_rows)
    header_detection = llm_analysis["header_detection"]
    detected_column_mapping = llm_analysis["column_mapping"]

    df = load_clean_dataframe(raw_df, header_detection["header_row_index"], detected_column_mapping["crypto"])
    resolved_column_mapping = resolve_column_mapping(df, detected_column_mapping)
    stats = compute_trade_stats(df, resolved_column_mapping)
    summary = build_trade_summary(stats, llm_analysis)

    return {
        "stats": stats,
        "summary": summary,
        "header_detection": header_detection,
        "column_mapping": resolved_column_mapping,
    }

In [11]:
# Example usage:
file_path = "/home/neosoft/Downloads/MehulAswar_0323202607130520260323-7-jpvg62.xlsx"
result = process_coindcx_file(file_path)
print(json.dumps(result["stats"], indent=2))
print(result["summary"])
print(result["header_detection"])
print(result["column_mapping"])

{
  "total_trades": 110,
  "buy_trades": 105,
  "sell_trades": 5,
  "total_buy_amount": 64932.784999999996,
  "avg_buy_price": 2813843.3490569075
}
Trade Summary
- Total trades: 110
- Buy trades: 105
- Sell trades: 5
- Total investment on buy side: INR 64,932.78
- Average buy price: INR 2,813,843.35

Key insights from the sampled rows:
- All visible trades are Buy orders; no Sell rows appear in the sample.
- Four cryptocurrencies are represented (SOL, ETH, BNB, BTC) with SOL trades showing a lower gross amount (~199 INR) compared to the others (~299 INR).
- Average prices differ dramatically across assets, reflecting their market values (e.g., SOL ~8.5k INR, ETH ~200k INR, BTC ~7 million INR).
{'header_row_index': 8, 'confidence': 0.99, 'reasoning': 'Row 8 contains descriptive column titles followed by trade data rows, matching typical header structure.'}
{'crypto': 'Crypto', 'side': 'Side (Buy/Sell)', 'gross_amount': 'Gross Amount Paid/Received by the user(in INR)', 'avg_price': 'Avg 